---
title: "6. Sound and music"
subtitle: "Speech, music, and sound design with generative audio models"
description: "Where machine listening stops and generation begins: how audio models represent sound, what speech, music, and sound design tools can do, what it takes to play one rather than prompt it, and whether anyone can still hear the difference."
---

This chapter works at the **practice** layer of [the five layers](intro.ipynb#five-layers). It begins where machine listening ends, at the turn from analysing sound to producing it. If you have spent any time computing spectrograms, extracting features, or training a classifier on audio, you already have most of the machinery this chapter needs. What changes is the direction you run it in.

Sound arrived late. Text and images reached the public in 2022, and audio of comparable quality took another year or two. By 2024 there were text-to-music services that produced finished songs from a sentence, and voice cloning that worked from half a minute of reference audio. By 2026 both sit inside ordinary podcast, film, and game workflows, which is why this chapter is about using them rather than about admiring them.

It is also the chapter where **consent** stops being abstract. A voice belongs to a body, it identifies the person it came from, and a convincing copy of it can be used to defraud, harass, or impersonate them. Chapter [3](co-creation-and-ethics.ipynb) sets out the general argument about training data and consent; this chapter is about what it means when the material is somebody's speaking voice and you are the one holding the microphone.

```{admonition} Question
:class: question
Before you read on, write down in one sentence what you think a music model is actually predicting while it generates. Keep it, and hold it against the chapter summary at the end.
```

## From analysis to generation

An analysis pipeline takes sound and turns it into something smaller: a spectrogram, a set of features, a sequence of codes, a label. A generative pipeline does the same journey in reverse. It produces the smaller thing and then reconstructs sound from it. The representation in the middle is the same object in both cases, which is why the two halves of this field are much closer than their separate literatures suggest.

Take the three representations you are most likely to have met. A **spectrogram** is a picture of how energy is distributed across frequency over time; an analysis system reads it, and a generative system writes one and then inverts it back to a waveform. A **neural audio codec** compresses sound into a short sequence of discrete codes; a classifier can be trained on those codes, and a generator can predict them and hand them to the codec's decoder. **Features** such as pitch, loudness, and spectral centroid summarise a signal for analysis, and the same curves work as control signals telling a synthesis model what to do next.

So the honest description of generative audio is not that it is a new field beside machine listening. It is machine listening run backwards through the same representation. That has a practical consequence for you: a good ear for what a representation throws away is also a good ear for what a generator will get wrong. A spectrogram discards phase, so a model that works on spectrograms has to have the phase invented for it on the way out, and that invention is exactly where the watery, smeared quality of some generated audio comes from.

The clearest demonstration of the turn is **RAVE**, a variational autoencoder for audio published as open research in 2021 [@Caillon2021]. It is trained on a corpus of sound and learns a compact latent representation of it, with an encoder that maps incoming audio into that space and a decoder that maps it back out. Train it on violin recordings, feed your own singing into the encoder, and the decoder reconstructs your gestures in the timbre of the corpus. That is **timbre transfer**: your phrasing, someone else's instrument.

What makes RAVE matter here is not the quality of any single output but its size. The model is small enough to run faster than real time on an ordinary laptop processor, which means the transfer happens while you sing rather than after you have finished. The weights and the training code are published, so you can train one on a corpus you recorded yourself, and researchers and musicians have. Chapter [2](how-it-works.ipynb) described the autoencoder in general; this is what one sounds like when you point it at your own voice.

## Three quick families of audio AI

At least three quite different things travel under the heading of generative audio, and confusing them is the most common way to pick the wrong tool.

1. **Text-to-speech and voice cloning.** A synthetic voice reading text you supply, optionally in the voice of a specific person captured from a short reference recording. Quality was transformed first by neural vocoders and then by transformer-based systems trained on very large speech corpora.
2. **Music generation.** Systems that produce a piece of music, often with vocals and lyrics, from a paragraph of prompt. The public research line runs through work such as MusicLM [@Agostinelli2023MusicLM]; the systems most students will actually use are commercial text-to-song services built on the same ideas.
3. **Sound effects and sound design.** Short generated events and atmospheres for film, game, and podcast work: footsteps on gravel, a door in a large hall, rain on a tin roof, an alien room tone.

A fourth family, speech recognition, is analysis rather than generation, so it belongs to the machine listening side of the boundary. It appears in this chapter only because it sits at the front of almost every practical audio workflow, including the one in the lab.

Underneath all three families is the pattern chapter [2](how-it-works.ipynb) set out for every generative model. A system trained on a large quantity of audio learns to predict how such audio tends to continue, and at inference time it produces something that resembles the distribution it was trained on. The differences between the families are differences of representation, of conditioning, and of what counts as a good result, not differences of principle.

```{figure} figures/sound/spectrogram.svg
:alt: A schematic spectrogram with frequency on the vertical axis and time on the horizontal axis.
:align: center
A spectrogram represents sound as an image, with time along the horizontal axis and frequency up the vertical one. Many audio models treat generating sound as generating an image of this kind and then converting it back.
```

## How models represent sound

Computers store sound as a long sequence of numbers, typically 44 100 or 48 000 of them per second per channel. That is an awkward quantity for a generative model. Producing audio directly, one sample at a time, is what the first widely known neural audio model did in 2016, and the results were remarkable and far too slow to use.

Modern audio models almost always work on a **compressed representation** instead. Two are common.

- A spectrogram, the time by frequency image above. Treat the image as an image, run a diffusion or transformer model over it, then convert the result back to a waveform with a *vocoder*, a network trained to do exactly that inversion.
- A **discrete code sequence** from a neural audio codec, a small autoencoder trained to compress audio into a handful of code streams and decode them again. The generative model predicts codes, and the codec turns them into sound.

Either route makes the sequence the model has to produce smaller by something like one to two orders of magnitude compared with raw samples. That reduction, more than any single architectural idea, is why a laptop can now produce a minute of music in seconds.

The backbone is usually a **transformer** [@Vaswani2017], adapted to the very long sequences audio implies, or a diffusion model operating on spectrograms or on codec tokens [@Agostinelli2023MusicLM]. The idea of learning a representation of sound rather than hand-designing one goes back further: a 2017 model trained an autoencoder on individual instrument notes and showed that you could interpolate smoothly between two timbres in the learned space, which is the ancestor of the latent-space instruments later in this chapter [@Engel2017NSynth].

Each representation has its own failure signature, and learning to hear them is a genuinely useful skill. Codec-based systems tend to produce artefacts that sound like low-bitrate streaming, a slight granularity or swirl in cymbals and sibilants. Spectrogram-based systems tend to smear transients, because a sharp attack is a narrow event in time that the vocoder has to reconstruct without the original phase. If you can tell those two apart by ear, you can usually tell which family of system produced a clip.

## Text-to-speech and voice cloning

A modern text-to-speech system takes two things: a piece of **text**, usually converted to phonemes on the way in, and a **speaker embedding**, a vector describing the desired voice. The embedding can come from a catalogue of stock voices or be extracted from a reference recording, and a few tens of seconds of clean speech is often enough. From those two the system produces a waveform.

What works well in 2026:

- **Convincing prosody** in English and the major European languages, Norwegian included with the right model.
- **Cloning a specific voice** from a short reference. This works considerably better than most people expect, which is the subject of the next section.
- **Style and emotion control**, either through a prompt, through tags in the text, or by supplying a reference clip in the delivery you want.
- **One voice across several languages**, without the speaker recording anything in the new language.

What still struggles:

- **Coherence over long durations.** A twenty-minute reading drifts in tone and energy, and the drift is easier to hear than to fix.
- **Singing and the boundary between speech and song.** Dedicated music systems handle this better than speech systems do.
- **Code-switching inside a sentence**, especially with technical terms, proper nouns, and abbreviations.
- **Low-resource languages.** Sámi languages, Faroese, and a great many African and Asian languages remain badly served, for the reasons chapter [4](language.ipynb) sets out about training data.

Norwegian is a partial exception on the recognition side, because of a deliberate public effort rather than commercial interest. The **NB AI Lab** at the National Library of Norway trains speech models on the library's own archive of recorded Norwegian and publishes the weights openly, and its NB-Whisper speech recognisers handle both written standards and a range of dialects while running on a laptop [@NBWhisper; @NBAILab]. Chapter [4](language.ipynb) makes the wider argument about why a small language needs open models rather than a hosted service.

### A note on consent

You can clone a recognisable version of someone's voice from a couple of minutes of publicly posted video. That is a statement of what is currently possible, not a suggestion.

**Cloning a real person's voice without their consent is harmful, and in a growing number of jurisdictions it is unlawful.** Treat a voice exactly as you would treat a face: get explicit permission, record what you were permitted to do with it, credit where credit is appropriate, and disclose that the result is synthetic. The EU AI Act places synthetic audio that imitates a real person under transparency obligations, and chapter [3](co-creation-and-ethics.ipynb) works through what those obligations mean in practice [@EUAIAct].

Two habits are worth forming now, because they cost nothing and are expensive to retrofit. Keep the consent in writing alongside the audio, saying who agreed to what. And keep a provenance note with every generated file, recording the source, the tool, and the date. The lab asks for both.

## Music generation

Music generation is harder than speech, for three reasons that are worth separating.

The **structure is longer-range**. A sentence is coherent over seconds; a song is coherent over minutes, across verses, choruses, a build, and a return. Predicting the next moment well does not make the third minute follow from the first.

The **judgement is aesthetic**. A wrong word is wrong in a way you can point at. A wrong note is a stylistic decision until proven otherwise, so the usual evaluation machinery has much less to bite on, and listening tests carry more of the weight than they can comfortably bear.

The **training data is contested**. Music is densely copyrighted, the recordings these systems learned from were generally used without the consent of the artists who made them, and in 2024 major labels took leading text-to-song services to court over exactly that [@RIAA2024]. As of 2026 the litigation is unresolved. The legal and practical questions are laid out carefully in a 2019 survey that has aged well [@Sturm2019], and the technical background to the whole field is covered in a book-length treatment [@Briot2020]. Artist-led responses exist too, including the Spawning coalition's work on letting creators opt their material out of training sets [@Spawning].

Despite all of that, as of 2026 a commercial text-to-song service reliably produces a two to three minute track from a paragraph, often with sung lyrics, and will usually let you export separated stems for further editing. The prompt elements that actually change the result are consistent across services:

- **Genre and era**, for example 1970s funk, Norwegian black metal, modern indie folk.
- **Instrumentation**, for example fingerpicked acoustic guitar, brushed snare, double bass.
- **Tempo and feel**, for example 110 beats per minute, swung eighths, intimate, late night.
- **Lyrics**, where supported, in their own field with section tags rather than mixed into the description.

What still fails is more interesting than what works, because it tells you where a human is still needed:

- **Named-artist imitation.** Asking for the style of a specific living artist raises the legal and ethical problems above, and is blocked outright by most services.
- **Lyrics outside English.** Norwegian output has improved and remains uneven, with stress patterns that a native speaker hears immediately.
- **Form beyond pop.** Multi-section classical works, fugal writing, and free improvisation are all much weaker than a three-minute verse-chorus song, which is the form these systems have seen most of.

The *Machine listening* chapter of *Sensing Sound and Music* covers how these systems learn and what the symbolic alternative looks like, so this chapter does not repeat it. The working question it leaves you with is this: the model is strong at idiomatic pastiche and weak at form, so which of those two does the piece you are making actually need?

## Sound design and Foley

Behind speech and music sits a quieter and more immediately useful category. **Sound design** models generate short events and atmospheres on request: five to fifteen seconds of rain on a tin roof, a wooden cart on cobblestones, a server room hum, a crowd in a hall two floors down.

This is the unglamorous workhorse of generative audio. It is less spectacular than song generation and much less morally fraught than voice cloning, because an atmosphere is not a person and rarely a recognisable copyrighted work. It is also the part of the field that has been absorbed into professional practice fastest, since a sound editor who needs a specific door in a specific room has always had to either record it or dig through a library, and now has a third option.

The craft has shifted rather than disappeared. Generated events are generic in exactly the way a stock library is generic, and making a scene sound like a place is still a matter of layering, placement, and what you leave out. Chapter [8](spatial.ipynb) takes up what happens when the same material has to sit in a three-dimensional scene.

## Playing with a model, not just prompting it

Everything above assumes the same interaction: you write something, you wait, you listen, you write something else. That is a compositional stance, and a reasonable one, but it is not what most musicians mean by playing. The difference is latency, and latency is not a detail here. It is what decides whether a model can be an instrument.

**Latency.** A text-to-song service takes tens of seconds to return a track, so you cannot respond to what you hear while it is happening; you can only judge the result and try again. An instrument works on a completely different timescale. Designers of digital instruments usually aim for a delay of the order of ten milliseconds between an action and the resulting sound, because beyond that the sound stops feeling caused by the gesture and starts feeling like a reply to it. Between those two timescales, seconds and milliseconds, lies the whole difference between prompting and playing.

**Real-time models.** Getting into the millisecond range means giving up size. A real-time audio model is small, trained on one specific corpus rather than everything, and built to process a short buffer of incoming audio and emit a short buffer of outgoing audio without ever seeing the future. RAVE is the standard example, and the reason it turns up in so many performances is that it hits the target on a laptop processor with no accelerator [@Caillon2021]. There are wrappers that run models of this kind inside the patching environments (visual programming tools for sound) and plugin hosts musicians already use, which matters more than it sounds: a model that lives in a notebook is a demonstration, and a model that lives in a plugin is a device on a stage.

**Instruments built on them.** Once the model runs in real time, its latent space becomes something you can reach into. The dimensions of that space are continuous controls, and they can be mapped to a fader, a pedal, a breath sensor, or the motion of a body, which turns the model into an instrument with a playing technique that has to be learned. The interesting design question stops being what to type and becomes what to map: which dimension goes to which gesture, and what the performer can therefore learn to do reliably. That is the same question chapter [12](body.ipynb) asks about movement and sensing, and it is why this chapter sits at the practice layer. You do not evaluate an instrument by looking at one output. You evaluate it by playing it for a week and asking what you can now do that you could not do before.

## Where sound AI fits in a real workflow

Three observations hold across the studios, newsrooms, and research groups currently using these tools.

**Generated audio is an ingredient, not a product.** It gets imported into a digital audio workstation, where it is layered, edited, equalised, compressed, and mixed against material that was recorded. Very little ships in the state the model produced it. Judging generative audio by the raw output is like judging photography by the unedited raw file.

**Stem separation changed what is possible downstream.** Models that split a finished mix into vocals, drums, bass, and everything else have become good enough for practical use, and the technique is set out in the [*Machine listening*](https://fourms.github.io/sensingsoundandmusic/machine-listening/) chapter of *Sensing Sound and Music*, which means you can take a generated track you cannot get clean stems out of and make your own. It is the quiet enabling technology behind a lot of remix and post-production work, and it belongs to the analysis side of the boundary rather than the generative one.

**Speech recognition is the silent revolution.** Automatic transcription went from an expensive service to a free local one in about five years, and researchers, journalists, and podcasters now use it daily without remarking on it. It is also the least ethically fraught audio tool in common use, because it produces text about a recording rather than a new recording of a person. It is the first step in this week's lab.

:::{note} Dig deeper: transcription and generation on your laptop
Both halves of the lab can be done on your own machine, which is worth trying at least once so that you know what depends on a service and what does not. Neither listing below is executed when this book is built.

Transcription with an open speech recogniser needs one install and one command. The Norwegian models from the National Library are drop-in replacements for the multilingual ones and are noticeably better on Norwegian audio [@NBWhisper].

```bash
pip install openai-whisper
whisper my-clip.mp3 --model small --language Norwegian
```

Music generation in code is heavier but still within reach of a machine with a modest graphics card, and a few minutes of patience without one. The `audiocraft` library from Meta's research group runs the MusicGen family locally.

```bash
pip install audiocraft
python -c "from audiocraft.models import MusicGen; MusicGen.get_pretrained('small')"
```

Running locally buys you three things a service cannot: the audio never leaves your machine, which matters for interview recordings and anything given to you in confidence; the model version is pinned, so this week's result is reproducible next year; and you can see exactly what the model was given.
:::

:::{admonition} Research spotlight
:class: spotlight
Two projects at [RITMO](https://www.uio.no/ritmo/), the interdisciplinary centre for rhythm, time, and motion at UiO, take generative sound out of headphones and put it in a room.

**MusicLab** is a series of research concerts in which the audience and the performers are measured while the music happens [@MusicLab]. Motion capture, heart rate, respiration, and questionnaires run alongside a real performance, and the data is analysed and often published openly afterwards. When generated sound appears in such a concert it stops being a file and becomes an event with bodies in it, which changes the questions you can ask: not whether a listener can identify a generated track in a survey, but what happens to breathing and movement in a hall when they hear one.

**The self-playing guitars** approach the same problem from the instrument side [@SelfPlayingGuitars]. Ordinary acoustic guitars are fitted with microcomputers and actuators that excite the strings directly, so a signal computed by a machine is radiated by a wooden body rather than by a loudspeaker. The sound is generated, but it reaches you as an instrument in a room, with the resonances, the directivity, and the slow decay that go with one. Put beside a real-time model such as RAVE, the pair sketches a research programme this chapter can only gesture at: what a generative model sounds like when it has a body.

Both run in Oslo and both appear at public events during the year, so attending one is realistic within the semester. If this is where your project is heading, say so in the lab session and we will point you at the people running them.
:::

:::{note} If you took MUS2640
Spectrograms, machine listening, music information retrieval, and a survey of generative music systems are all covered in the [*Machine listening*](https://fourms.github.io/sensingsoundandmusic/machine-listening/) chapter of *Sensing Sound and Music*, and this chapter deliberately starts where that one ends.

So *From analysis to generation* above is the bridge rather than new material, and the sections on representation will be revision. Start at *Text-to-speech and voice cloning* if your time is short, and read *Playing with a model, not just prompting it* carefully, because real-time generation is the part that chapter does not cover.

Use the [live spectrogram app](https://fourms.github.io/sensingsoundandmusic/apps/live-spectrogram/) from that book for the listening exercises here rather than looking for a second one. It is the same representation, and there is no reason to learn a new interface for it.
:::

:::{note} If you took MUS2850
*Computer Music* teaches sound synthesis, MIDI, and digital signal processing through audio-specific graphical programming languages such as Max/MSP, Pure Data, and SuperCollider. MIDI is the symbolic protocol that carries notes and controller values rather than sound. Digital signal processing is the mathematics of filtering, delaying, and otherwise transforming a sampled waveform. You arrive here with signal thinking, with patching experience, and with the habit of keeping control data apart from audio.

Read a generative audio model as a very large synthesiser whose parameters were learned from recordings rather than set by hand. Symbolic generation produces something MIDI-like, a stream of notes and control values, while audio generation produces the waveform itself. Latency and control are the same problems you met in the workshops, so *Playing with a model, not just prompting it* is where your instincts pay off.

If you have not taken MUS2850, nothing in this chapter requires it. It is the natural place to go afterwards for the synthesis craft, and the two courses work in either order.
:::

## This week's lab: Explore, Reflect, Create

This lab runs the chapter's own turn in miniature: analyse a recording, generate from it, and then argue about what the generation lost.

### Explore (about 30 min)

**Transcribe and re-voice.** Work with a recording you have the right to use.

1. Record, or choose, a 30 to 60 second clip of speech. If it is somebody else's voice, get their permission first and write down what they agreed to. Your own voice is the simplest option and makes the consent question easy.
2. Transcribe it with a speech recogniser, either locally using the listing in the Dig deeper note above or through a web service. Note how long it took and how many corrections the transcript needs.
3. Generate a new synthetic voice reading the same text, using a stock voice rather than a clone of anyone.
4. Put the two side by side and listen twice: once for what the transcription got wrong, and once for what the synthetic reading adds and removes. Pay particular attention to breaths, hesitations, and emphasis, because those are where the difference lives.
5. Record the comparison in your log, with the tools, the models, and the date.

**Optional.** Run both the original and the synthetic version through the [live spectrogram](https://fourms.github.io/sensingsoundandmusic/apps/live-spectrogram/) from *Sensing Sound and Music* and see whether the difference you heard is visible.

### Reflect (about 15 min)

Work in pairs, then in plenary. This is a discussion, not a writing block.

1. Play each other a 30-second generated music clip, without saying where it came from. Each of you says what gives it away and, harder, what does not. Try to name the cue: is it the mix, the vocal, the drums, the way the section changes, or something you cannot locate?
2. Take one position each on voice cloning as a free and universally available service, and argue it properly for three minutes: what changes for journalism, for political advertising, for a person whose voice is their livelihood, for your own recordings.
3. Close the round by stating aloud the brief for the piece you are about to make: what it is for, how long, and what it has to do. Say it as a job, not as a mood.

### Create (about 45 min)

**A 30-second piece.** Make one finished thirty seconds of audio combining generated music and a generated ambience bed.

1. Take the brief you stated in the Reflect round, for example thirty seconds of background music for a research lab promotional video.
2. Generate music with a text-to-song service and iterate the prompt until you have something usable. Keep every prompt you tried, not just the one that worked.
3. Generate a separate ambience or sound-effect bed, and treat it as a layer rather than a decoration.
4. Mix the two in any digital audio workstation, including a free one. Set levels, cut what you do not need, and give the thirty seconds a shape.
5. Write a **consent and provenance note**: every tool and model with its version, the prompts, the edits you made by hand, the time taken, and for any voice or recorded sample, where it came from and whether you have the right to publish it.
6. Export the mix and commit it to your portfolio with the note beside it.

**A3 starts this week.** *Multimodal mini-piece* is set now and due in week 8: three to five pages or slides combining text and at least one other modality, plus a reflection. The thirty-second piece is a candidate component of A3, not A3 itself. If you want the audio to end up in your submission, decide this week what the audio is *for*, because a sound bed with no piece attached is much harder to place in week 8 than in week 6.

At home, write this week's entry in your practice log using the [practice log template](templates/practice-log.md).

## A critical look: can listeners tell AI music from human music?

**The claim.** Listeners can no longer distinguish generated music from music made by people, so the distinction has stopped mattering.

**The evidence.** The headline number comes from an industry survey published in November 2025 by a streaming service and a polling company, which reported that the overwhelming majority of respondents could not correctly identify fully generated tracks when they heard them [@DeezerIpsos2024]. Taken at face value that is a striking result, and it was reported as one. Smaller controlled studies suggest a more qualified picture, in which trained listeners tend to do better than the general population and longer excerpts help everyone, though that literature is thin beside the headline survey.

**The method.** Look at how the striking number is produced. Respondents hear short excerpts, usually thirty seconds, usually pop, usually through whatever hardware they happen to have and often at streaming bitrates, and they make a forced choice between two options. Every one of those decisions makes detection harder. Thirty seconds of pop is the format generative systems are strongest at and the format in which a human production is most conventional, so the two are being compared in the narrow region where they are most alike. A forced choice also converts uncertainty into a guess, which is not the same as being fooled. And a survey run by a company that also announces how much generated music it is filtering out is not a neutral instrument, whatever the polling partner's standards.

**The limits.** Short pop excerpts are the easy case for a generator, and the easy case is being reported as the general case. The weaknesses are elsewhere. Long-range form is where these systems are demonstrably weak, because coherence across four minutes is not what a model trained to predict the next moment optimises for, and a thirty-second window is exactly the window in which that weakness cannot show. Deliberate rule-breaking is the other gap: a model trained to produce what is likely will not produce the choice that is unlikely and right, which is a fair description of most of what changes a style. The *Machine listening* chapter of *Sensing Sound and Music* reaches the same conclusion from the analysis side, and both limits, along with the questions of law and practice that come with them, are long-standing in the literature [@Briot2020; @Sturm2019].

There is also a question the survey format cannot ask. "Can you tell?" is not "does it hold up?", and a track that survives a forced choice at thirty seconds may still be one nobody returns to. The useful version is narrower and answerable: for the job you have, at the length you need, with the listeners you have, does the difference matter? That has a different answer for a fifteen-second background bed than for a piece somebody is expected to sit and listen to, and you can settle it in this week's lab.

```{admonition} Chapter summary
:class: tip
Generative audio is machine listening run backwards through the same representations, so a spectrogram, a codec's code sequence, or a set of features is written by a generator rather than read by an analyser. Three families sit under the heading: text-to-speech and voice cloning, music generation, and sound design, and they differ in representation, conditioning, and what counts as success rather than in principle. Models work on compressed representations because raw samples are too many, which makes generation practical and leaves each representation its own audible failure signature. Voice cloning works from a short reference, which makes consent the central practical obligation of this chapter and provenance a habit rather than a formality. Music generation is strong at idiomatic pastiche and weak at long-range form and deliberate rule-breaking, and its training data is contested in court. Latency is what separates prompting from playing: small real-time models such as RAVE run fast enough on a laptop to become instruments with latent spaces you can map to gestures. In practice generated audio is an ingredient that goes into a mix, not a finished product, and the question worth asking is not whether a listener can tell but whether the difference matters for the job in hand.
```

```{admonition} Questions
:class: question
1. Name the three families of generative audio in this chapter and give one task from your own field where each would be the right choice.
2. Why do audio models work on spectrograms or codec tokens rather than on raw samples, and what does each of those two representations tend to get wrong?
3. What does consent mean when the material is a person's voice? Say what you would need to have in writing before publishing a cloned voice, and what you would disclose to the listener.
4. What is music generation still bad at, and why does the length of the excerpt used in a listening test change the answer?
5. Where does generated audio actually sit in a working pipeline, and what would you have to do to a generated thirty-second bed before it could ship?
```

:::{seealso} Further reading
- **Caillon and Esling, *RAVE: A variational autoencoder for fast and high-quality neural audio synthesis* (2021)** — the model behind most real-time neural audio performance, and readable if you take the autoencoder section slowly [@Caillon2021]. [arxiv.org/abs/2111.05011](https://arxiv.org/abs/2111.05011)
- **Briot, Hadjeres and Pachet, *Deep Learning Techniques for Music Generation* (2020)** — the book-length survey of the field, organised by what you want to generate rather than by architecture [@Briot2020]. [doi.org/10.1007/978-3-319-70163-9](https://doi.org/10.1007/978-3-319-70163-9)
- **Sturm et al., *Artificial Intelligence and Music: Open Questions of Copyright Law and Engineering Praxis* (2019)** — written before the current wave and still the clearest statement of the legal questions it raised [@Sturm2019]. [doi.org/10.3390/arts8030115](https://doi.org/10.3390/arts8030115)
- **Agostinelli et al., *MusicLM: Generating Music From Text* (2023)** — the research paper that set the template for text-to-music, with audio examples worth listening to critically [@Agostinelli2023MusicLM]. [arxiv.org/abs/2301.11325](https://arxiv.org/abs/2301.11325)
- **Engel et al., *Neural Audio Synthesis of Musical Notes with WaveNet Autoencoders* (2017)** — early, influential, and the origin of the idea that you can interpolate between timbres in a learned space [@Engel2017NSynth]. [arxiv.org/abs/1704.01279](https://arxiv.org/abs/1704.01279)
- **NB-Whisper, from the NB AI Lab at the National Library of Norway** — open Norwegian speech recognition that runs on a laptop, and the model to reach for in the lab [@NBWhisper; @NBAILab]. [huggingface.co/NbAiLab/nb-whisper-large](https://huggingface.co/NbAiLab/nb-whisper-large)
- **The Spawning coalition** — artist-led tools for opting voices, likeness, and work out of training datasets [@Spawning]. [spawning.ai](https://spawning.ai/)

For the synthesis and audio-programming craft underneath all of this, the deepening route at UiO is [MUS2850 Computer Music](https://www.uio.no/studier/emner/hf/imv/MUS2850/), taught each autumn and open in either order with this course.
:::

:::{tip} Explore interactively
- [Markov melody generator](https://fourms.github.io/Creative-AI/apps/markov-melody/): build a melody generator from a handful of tunes and hear what a model that only knows what usually comes next can and cannot do.
- [Live spectrogram](https://fourms.github.io/sensingsoundandmusic/apps/live-spectrogram/), from *Sensing Sound and Music*: sing, speak, or play into your microphone and watch the representation this chapter's models are writing into.
:::